Q1 — What does lateness cost in customer satisfaction?

In [0]:
%sql
SELECT
  CASE
    WHEN days_late <= -5 THEN 'Early (5+ days)'
    WHEN days_late <   0 THEN 'Early (1-4 days)'
    WHEN days_late =   0 THEN 'On time'
    WHEN days_late <=  3 THEN 'Late (1-3 days)'
    WHEN days_late <= 10 THEN 'Late (4-10 days)'
    ELSE 'Late (10+ days)'
  END                       AS delivery_bucket,
  COUNT(*)                  AS orders,
  ROUND(AVG(review_score),2) AS avg_review,
  ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 1) AS pct_1_2_star
FROM olist.gold.fact_order_delivery
WHERE review_score IS NOT NULL
GROUP BY 1
ORDER BY MIN(days_late);

Q2 — Which sellers need intervention?

In [0]:
%sql
SELECT primary_seller_id, seller_state, orders,
       on_time_pct, avg_delivery_days, avg_review, revenue
FROM olist.gold.agg_seller_performance
WHERE on_time_pct < 85
ORDER BY revenue DESC
LIMIT 20;

Q3 — Is it getting better or worse?

In [0]:
%sql
SELECT order_month, orders, late_pct, avg_delivery_days, avg_review
FROM olist.gold.agg_monthly_trend
WHERE orders > 100
ORDER BY order_month;

Q5 — Data quality over time (for the appendix slide)

In [0]:
%sql
SELECT DATE(checked_at) AS day,
       SUM(CASE WHEN passed THEN 1 ELSE 0 END) AS passed,
       SUM(CASE WHEN NOT passed THEN 1 ELSE 0 END) AS failed
FROM olist.gold.dq_audit
WHERE severity IN ('critical','warn')
GROUP BY 1 ORDER BY 1;